# COTD: **Calories of The Day - Agent**



## App Objectives  


## Roadmap
## 1. EXECUTIVE SUMMARY
 
A multi-modal calorie tracking agent that accepts **food photos or text descriptions** and uses local reasoning-capable LLMs to:
- Identify ingredients intelligently through image analysis or text parsing
- Gather missing nutritional information via agentic questioning
- Query a nutrition API for macro/micronutrient data
- Store results with metadata (timestamp, date, meal type)
- Run entirely on local hardware using 3B parameter models
 
The system **must reason about completeness** (do we have enough ingredient data?) and **fail gracefully** (use averages if user skips/refuses detailed input).
 
---
 
## 2. SYSTEM ARCHITECTURE
 
### 2.1 High-Level Flow
 
```
USER INPUT (photo OR text)
    ↓
[ROUTER AGENT] - Decide: Image or Text?
    ↓
    ├─→ IMAGE AGENT (if photo)
    │   ├─ Vision analysis
    │   ├─ Ingredient extraction
    │   └─ Agentic questioning loop
    │
    └─→ TEXT AGENT (if description)
        ├─ NLP ingredient parsing
        └─ Agentic questioning loop
    ↓
[REASONING AGENT] - Check completeness
    ├─ Do we have all ingredient amounts?
    ├─ Are units standardized?
    └─ Is reasoning sufficient to proceed?
    ↓
[NUTRITION LOOKUP AGENT] - Fetch data
    ├─ Query USDA FoodData Central API
    ├─ Aggregate macro/micronutrients
    └─ Handle missing entries (fallback to generics)
    ↓
[DATABASE AGENT] - Record & Persist
    ├─ Timestamp, meal type, date
    ├─ All ingredients + amounts
    ├─ Nutritional breakdown
    └─ User session metadata
    ↓
SUMMARY & VISUALIZATION

# 0- Environment setup

### Step 1: Create your environment and install dependencies 
Before we start coding, you need a reproducible setup. Open a terminal in the same directory as this notebook, and use Conda or uv to install the project dependencies.

#### Option 2: UV (faster)

If you prefer [uv](https://docs.astral.sh/uv/) over Conda:

```bash
# Install uv (skip if already installed)
curl -LsSf https://astral.sh/uv/install.sh | sh

# Create venv and install dependencies
uv venv .venv --python 3.11 && source .venv/bin/activate
uv pip install -r requirements.txt
```

### Step 2: Register this environment as a Jupyter kernel
```bash
python -m ipykernel install --user --name=web_agent --display-name "web_agent"
```
Now open your notebook and switch to the `web_agent` kernel (Kernel → Change Kernel).

### Step 3: Set up Ollama

In this project, we use **Ollama** to load and use open-weight LLMs. We start with smaller models like `gemma3:1b` and then switch to larger models like `llama3.2:3b`.

Start the **Ollama** server in a terminal. This launches a local API endpoint that listens for LLM requests.

```bash
ollama serve
```

Downloads the model so you can run them locally without API calls. 
```bash
ollama pull gemma3:1b
ollama pull llama3.2:3b
```

You can explore other available models [here](https://ollama.com/library) and pull them to experiment with.

In [1]:
# Quick check: is Ollama running?
# If this fails, open a terminal and run: ollama serve

import httpx

response = httpx.get("http://localhost:11434/api/tags", timeout=5)
models = [m["name"] for m in response.json().get("models", [])]
print(f"Ollama is running. Installed models: {models}")

Ollama is running. Installed models: ['llama3.2:3b', 'qwen2.5:3b-instruct', 'deepseek-r1:1.5b', 'gemma3:1b']


# 1 — Data Structures

Define the core dataclasses used throughout all agents. These represent the shared state passed between agents.

In [2]:
from __future__ import annotations
from dataclasses import dataclass, field
from datetime import datetime, date
from typing import Optional, List
import json

@dataclass
class NutritionData:
    calories: float
    protein_g: float
    fat_g: float
    carbs_g: float
    fat_saturated_g: Optional[float] = None
    fiber_g: Optional[float] = None
    sugar_g: Optional[float] = None
    sodium_mg: Optional[float] = None
    vitamin_a_mcg: Optional[float] = None
    vitamin_c_mg: Optional[float] = None
    calcium_mg: Optional[float] = None
    iron_mg: Optional[float] = None
    api_source: str = "unknown"
    confidence: float = 1.0

@dataclass
class Ingredient:
    name: str
    amount: float
    unit: str
    confidence: float = 1.0
    source: str = "user_input"          # user_input | visual_estimate | fallback_average
    cooking_method: Optional[str] = None
    notes: Optional[str] = None
    nutrition: Optional[NutritionData] = None

@dataclass
class Meal:
    meal_type: str                      # breakfast | lunch | dinner | snack
    ingredients: List[Ingredient] = field(default_factory=list)
    meal_timestamp: datetime = field(default_factory=datetime.now)
    meal_date: date = field(default_factory=date.today)
    total_calories: float = 0.0
    total_protein_g: float = 0.0
    total_fat_g: float = 0.0
    total_carbs_g: float = 0.0
    total_fiber_g: float = 0.0
    mood: Optional[str] = None
    notes: Optional[str] = None
    api_fallback_count: int = 0
    id: Optional[int] = None

print("✓ Data structures defined: NutritionData, Ingredient, Meal")

✓ Data structures defined: NutritionData, Ingredient, Meal


In [3]:
# --- Test: Data Structures ---
n = NutritionData(calories=245, protein_g=45.0, fat_g=5.0, carbs_g=0.0, api_source="USDA_FDC")
i = Ingredient(name="chicken breast", amount=145, unit="g", cooking_method="grilled", nutrition=n)
m = Meal(meal_type="lunch", ingredients=[i])

assert i.name == "chicken breast"
assert i.nutrition.calories == 245
assert m.meal_type == "lunch"
print("✓ Data structure test passed")

✓ Data structure test passed


# 2 — Ollama Helper

A thin wrapper around Ollama's HTTP API used by all agents.

In [4]:
import httpx, re

OLLAMA_URL = "http://localhost:11434/api/chat"

def ollama_chat(model: str, messages: list[dict], temperature: float = 0.0) -> str:
    """Call Ollama chat endpoint and return the assistant message content."""
    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {"temperature": temperature},
    }
    resp = httpx.post(OLLAMA_URL, json=payload, timeout=120)
    resp.raise_for_status()
    return resp.json()["message"]["content"].strip()

def extract_json(text: str) -> dict:
    """Pull the first JSON object out of a model response."""
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON found in model output:\n{text}")
    return json.loads(match.group())

print("✓ Ollama helper ready")

✓ Ollama helper ready


In [5]:
# --- Test: Ollama helper ---
reply = ollama_chat("gemma3:1b", [{"role": "user", "content": "What model are you?"}])
print(f"Model replied: {reply}")
print("✓ Ollama helper test passed")

Model replied: I’m Gemma, a large language model created by the Gemma team at Google DeepMind. I’m an open-weights model, which means I’m widely available for public use!
✓ Ollama helper test passed


# 3 — Router Agent

Classifies user input as `TEXT_AGENT` or `IMAGE_AGENT`. Uses `gemma3:1b` for fast, cheap classification.

In [6]:
ROUTER_MODEL = "gemma3:1b"

ROUTER_SYSTEM = """You are a food input router. Classify user input into exactly one category:
- IMAGE_AGENT  → user provides an image file path, URL ending in .jpg/.png, or explicitly mentions a photo
- TEXT_AGENT   → user describes food in text (ingredients, meal description, etc.)

Respond with ONLY one of these two words: IMAGE_AGENT or TEXT_AGENT"""

def router_agent(user_message: str, has_image: bool = False) -> str:
    """Return 'IMAGE_AGENT' or 'TEXT_AGENT'."""
    if has_image:
        return "IMAGE_AGENT"
    
    reply = ollama_chat(
        ROUTER_MODEL,
        [
            {"role": "system", "content": ROUTER_SYSTEM},
            {"role": "user", "content": user_message},
        ],
    )
    
    if "IMAGE_AGENT" in reply.upper():
        return "IMAGE_AGENT"
    return "TEXT_AGENT"

print("✓ Router agent defined")

✓ Router agent defined


In [7]:
# --- Test: Router Agent ---
cases = [
    ("100g grilled chicken and a side salad", False, "TEXT_AGENT"),
    ("Here is my lunch photo: meal.jpg", False, "IMAGE_AGENT"),
    ("I had a banana and some yogurt", False, "TEXT_AGENT"),
    ("anything", True, "IMAGE_AGENT"),   # has_image flag overrides
]

for msg, has_img, expected in cases:
    result = router_agent(msg, has_image=has_img)
    status = "✓" if result == expected else "✗"
    print(f"{status} '{msg[:40]}...' → {result} (expected {expected})")
print("✓ Router agent tests complete")

✓ '100g grilled chicken and a side salad...' → TEXT_AGENT (expected TEXT_AGENT)
✗ 'Here is my lunch photo: meal.jpg...' → TEXT_AGENT (expected IMAGE_AGENT)
✓ 'I had a banana and some yogurt...' → TEXT_AGENT (expected TEXT_AGENT)
✓ 'anything...' → IMAGE_AGENT (expected IMAGE_AGENT)
✓ Router agent tests complete


# 4 — Text Agent

Parses free-form food descriptions into structured `Ingredient` objects using `qwen2.5:3b-instruct`. Returns a list of ingredients with name, amount, unit, and cooking method.

In [8]:
TEXT_AGENT_MODEL = "qwen2.5:3b-instruct"

TEXT_AGENT_SYSTEM = """You are a food ingredient parser. Extract all food ingredients from the user's message.

Rules:
- "name" = the food item ONLY (e.g. "banana", "greek yogurt", "honey"). NEVER include measurement words like "cup", "tbsp", "tsp", "g" in the name.
- "amount" = numeric value (float). If unknown, use null.
- "unit" = "g", "ml", "piece", "cup", "tbsp", "tsp". If unknown, use null.
- "cooking_method" = "grilled", "raw", "steamed", etc. If unknown, use null.
- if you don't know the unit or the food is 99% that the amount is 0, do not include in the output.

EXAMPLE:
Input: "1 banana, 1 cup Greek yogurt, 1 tbsp honey"
Output:
{
  "ingredients": [
    {"name": "banana", "amount": 1, "unit": "piece", "cooking_method": null},
    {"name": "greek yogurt", "amount": 1, "unit": "cup", "cooking_method": null},
    {"name": "honey", "amount": 1, "unit": "tbsp", "cooking_method": null}
  ]
}

EXAMPLE (Chinese input → English output):
Input: "一个香蕉，一杯希腊酸奶，没吃蜂蜜"
Output:
{
  "ingredients": [
    {"name": "banana", "amount": 1, "unit": "piece", "cooking_method": null},
    {"name": "greek yogurt", "amount": 1, "unit": "cup", "cooking_method": null}
  ]
}

Respond ONLY with valid JSON. No markdown, no explanation."""

def text_agent(user_message: str) -> list[Ingredient]:
    """Parse a text food description into a list of Ingredient objects."""
    reply = ollama_chat(
        TEXT_AGENT_MODEL,
        [
            {"role": "system", "content": TEXT_AGENT_SYSTEM},
            {"role": "user", "content": user_message},
        ],
    )

    # Strip markdown fences in case model adds them
    import re
    reply = re.sub(r'```(?:json)?\s*', '', reply)
    reply = re.sub(r'```', '', reply)

    parsed = extract_json(reply)
    ingredients = []
    for item in parsed.get("ingredients", []):
        ingredients.append(Ingredient(
            name=item["name"],
            amount=float(item["amount"]) if item.get("amount") is not None else 0.0,
            unit=item.get("unit") or "unknown",
            cooking_method=item.get("cooking_method"),
            confidence=0.9 if item.get("amount") is not None else 0.3,
            source="user_input",
        ))
    return ingredients

print("✓ Text agent defined")

✓ Text agent defined


In [9]:
# --- Test: Text Agent ---
test_input = "我吃了一碗青椒炒肉丝，一磅猪肉里脊，没吃青椒"
ingredients = text_agent(test_input)

print(f"Parsed {len(ingredients)} ingredients:")
for ing in ingredients:
    print(f"  • {ing.name}: {ing.amount} {ing.unit}  [method={ing.cooking_method}, conf={ing.confidence}]")

names = [i.name for i in ingredients]
print("✓ Text agent test passed")

Parsed 1 ingredients:
  • pork loin: 1.0 lb  [method=cooked, conf=0.9]
✓ Text agent test passed


# 5 — Reasoning Agent (Completeness Checker)

Checks if the ingredient list is complete enough to proceed (score ≥ 0.75). If not, generates one targeted clarifying question. Uses `deepseek-r1:1.5b` for step-by-step reasoning.

In [27]:
REASONING_MODEL = "deepseek-r1:1.5b"

REASONING_SYSTEM = """You are a meal completeness checker. Given a list of food ingredients, evaluate if we have enough data for accurate calorie calculation.

For EACH ingredient assess:
- Is the name specific enough?
- Is there a numeric amount? (null = missing = low confidence)
- Is the unit standard? ("handful" or "some" = vague)
- For proteins and oils: is cooking method specified?

Scoring per ingredient:
- Amount missing OR unit vague → confidence 0.3
- Non-standard unit (handful, some, a bit) → confidence 0.6
- Missing cooking method for protein/oil → confidence 0.8
- Everything clear → confidence 1.0

overall_completeness = mean of all ingredient confidences

Respond ONLY with valid JSON:
{
  "ingredient_scores": [{"name": "...", "confidence": 0.0, "issue": "...or null"}],
  "completeness_score": 0.0,
  "next_question": "...one specific question, or null"
}

CRITICAL RULE: if completeness_score < 0.75 you MUST write a question in next_question. It must NEVER be null when score < 0.75."""

def reasoning_agent(ingredients: list[Ingredient]) -> dict:
    ing_list = [
        {
            "name": i.name,
            "amount": i.amount if i.amount > 0 else None,
            "unit": i.unit,
            "cooking_method": i.cooking_method,
        }
        for i in ingredients
    ]

    reply = ollama_chat(
        REASONING_MODEL,
        [
            {"role": "system", "content": REASONING_SYSTEM},
            {"role": "user", "content": f"Ingredients: {json.dumps(ing_list)}"},
        ],
    )

    result = extract_json(reply)

    # Override with our own completeness calculation — small model is unreliable.
    def _ing_confidence(ing):
        if ing.amount <= 0: return 0.3
        if ing.unit not in UNIT_TO_GRAMS: return 0.6
        return 1.0
    real_score = sum(_ing_confidence(i) for i in ingredients) / len(ingredients) if ingredients else 1.0
    result["completeness_score"] = max(result.get("completeness_score", 0.0), real_score)

    if result["completeness_score"] < 0.75:
        needs_info = [i for i in ingredients if i.amount <= 0 or i.unit not in UNIT_TO_GRAMS]
        if needs_info:
            target = needs_info[0]
            if target.amount <= 0:
                result["next_question"] = f"How much {target.name} did you have? (e.g. 1 cup, 200g, 1 plate)"
            else:
                result["next_question"] = f"Can you be more specific about how much {target.name}? (e.g. grams or cups)"
        else:
            result["completeness_score"] = 1.0
            result["next_question"] = None
    return result

print("✓ Reasoning agent defined")


✓ Reasoning agent defined


In [28]:
# --- Test: Reasoning Agent ---

# Case 1: complete ingredients — should proceed (no question)
complete_ings = [
    Ingredient("chicken breast", 145, "g", cooking_method="grilled"),
    Ingredient("broccoli", 90, "g", cooking_method="steamed"),
    Ingredient("olive oil", 5, "ml"),
]
result_complete = reasoning_agent(complete_ings)
print("Complete meal check:")
print(f"  score={result_complete['completeness_score']:.2f}  question={result_complete['next_question']}")
assert result_complete["completeness_score"] >= 0.7, "Complete meal should score ≥ 0.7"

# Case 2: missing amounts — should ask a question
incomplete_ings = [
    Ingredient("pasta", 0.0, "unknown"),
    Ingredient("tomato sauce", 0.0, "unknown"),
]
result_incomplete = reasoning_agent(incomplete_ings)
print("\nIncomplete meal check:")
print(f"  score={result_incomplete['completeness_score']:.2f}  question={result_incomplete['next_question']}")
assert result_incomplete["next_question"] is not None, "Incomplete meal should generate a question"

print("\n✓ Reasoning agent tests passed")

Complete meal check:
  score=1.00  question=None

Incomplete meal check:
  score=0.30  question=How much pasta did you have? (e.g. 1 cup, 200g, 1 plate)

✓ Reasoning agent tests passed


# 6 — USDA FoodData Central — Nutrition Lookup

Queries the USDA FDC API for macronutrient data, scales to user portion, and caches results in SQLite to minimize repeat API calls.

> **API key:** USDA provides a `DEMO_KEY` that allows ~30 req/hr — sufficient for development. No sign-up needed.

In [12]:
import sqlite3, requests, os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # loads .env from project root
USDA_API_KEY = os.getenv("USDA_API_KEY", "DEMO_KEY")
USDA_SEARCH_URL = "https://api.nal.usda.gov/fdc/v1/foods/search"
DB_PATH = Path("cotd.db")

# Nutrient IDs in USDA FDC
NUTRIENT_IDS = {
    "calories":  1008,  # Energy (kcal)
    "protein_g": 1003,  # Protein
    "fat_g":     1004,  # Total lipid (fat)
    "carbs_g":   1005,  # Carbohydrate, by difference
    "fiber_g":   1079,  # Fiber, total dietary
    "sugar_g":   2000,  # Sugars, total
    "sodium_mg": 1093,  # Sodium
    "fat_saturated_g": 1258,  # Fatty acids, total saturated
}

# Generic fallbacks (kcal/100g) when USDA has no match
FALLBACK_GENERICS = {
    "vegetable": NutritionData(calories=30, protein_g=2, fat_g=0.3, carbs_g=5, api_source="fallback_generic", confidence=0.3),
    "fruit":     NutritionData(calories=60, protein_g=0.8, fat_g=0.2, carbs_g=15, api_source="fallback_generic", confidence=0.3),
    "meat":      NutritionData(calories=150, protein_g=26, fat_g=5, carbs_g=0, api_source="fallback_generic", confidence=0.3),
    "grain":     NutritionData(calories=130, protein_g=4, fat_g=1, carbs_g=27, api_source="fallback_generic", confidence=0.3),
    "oil":       NutritionData(calories=884, protein_g=0, fat_g=100, carbs_g=0, api_source="fallback_generic", confidence=0.3),
    "dairy":     NutritionData(calories=61, protein_g=3.2, fat_g=3.3, carbs_g=4.8, api_source="fallback_generic", confidence=0.3),
    "default":   NutritionData(calories=100, protein_g=3, fat_g=3, carbs_g=12, api_source="fallback_generic", confidence=0.2),
}

# Unit → grams/ml conversion table
UNIT_TO_GRAMS = {
    "g": 1.0, "ml": 1.0, "kg": 1000.0, "l": 1000.0,
    "oz": 28.35, "lb": 453.6,
    "cup": 240.0, "tbsp": 15.0, "tsp": 5.0,
    "piece": 100.0, "medium": 100.0, "large": 150.0, "small": 70.0,
    "unknown": 100.0,
}

def unit_to_grams(amount: float, unit: str) -> float:
    factor = UNIT_TO_GRAMS.get(unit.lower(), 100.0)
    return amount * factor

def _extract_nutrients(food_item: dict) -> NutritionData:
    """Pull relevant nutrients from a USDA food item dict."""
    nutrients = {n["nutrientId"]: n.get("value", 0.0) for n in food_item.get("foodNutrients", [])}
    return NutritionData(
        calories=nutrients.get(NUTRIENT_IDS["calories"], 0.0),
        protein_g=nutrients.get(NUTRIENT_IDS["protein_g"], 0.0),
        fat_g=nutrients.get(NUTRIENT_IDS["fat_g"], 0.0),
        carbs_g=nutrients.get(NUTRIENT_IDS["carbs_g"], 0.0),
        fiber_g=nutrients.get(NUTRIENT_IDS["fiber_g"]),
        sugar_g=nutrients.get(NUTRIENT_IDS["sugar_g"]),
        sodium_mg=nutrients.get(NUTRIENT_IDS["sodium_mg"]),
        fat_saturated_g=nutrients.get(NUTRIENT_IDS["fat_saturated_g"]),
        api_source="USDA_FDC",
        confidence=0.95,
    )

def _scale(base: NutritionData, grams: float) -> NutritionData:
    """Scale per-100g nutrition to the actual gram amount."""
    ratio = grams / 100.0
    return NutritionData(
        calories=round(base.calories * ratio, 1),
        protein_g=round(base.protein_g * ratio, 2),
        fat_g=round(base.fat_g * ratio, 2),
        carbs_g=round(base.carbs_g * ratio, 2),
        fiber_g=round(base.fiber_g * ratio, 2) if base.fiber_g else None,
        sugar_g=round(base.sugar_g * ratio, 2) if base.sugar_g else None,
        sodium_mg=round(base.sodium_mg * ratio, 1) if base.sodium_mg else None,
        fat_saturated_g=round(base.fat_saturated_g * ratio, 2) if base.fat_saturated_g else None,
        api_source=base.api_source,
        confidence=base.confidence,
    )

_PROCESSED = {"breaded", "fried", "dehydrated", "powder", "dried",
              "frozen", "canned", "mix", "flavored", "instant", "flour", "unenriched"}

def _usda_search(query: str) -> NutritionData | None:
    """Query USDA and return per-100g NutritionData, or None on failure."""
    try:
        r = requests.get(
            USDA_SEARCH_URL,
            params={"query": query, "pageSize": 20, "api_key": USDA_API_KEY,
                    "dataType": ["Foundation", "SR Legacy"]},
            timeout=10,
        )
        if r.status_code == 429:
            print("  [USDA] Rate limit hit — replace DEMO_KEY with a free key at fdc.nal.usda.gov/api-key-signup.html")
            return None
        r.raise_for_status()
        foods = r.json().get("foods", [])
        if not foods:
            return None
        q_words = query.lower().split()
        def _score(food):
            desc = food.get("description", "").lower()
            hits    = sum(1 for w in q_words if w in desc)
            first   = 2 if any(desc.startswith(w) for w in q_words) else 0
            penalty = -3 * len(_PROCESSED & set(desc.split(",")[0].split()))
            return (first + hits + penalty, -len(desc))
        best = max(foods, key=_score)
        return _extract_nutrients(best)
    except Exception as e:
        print(f"  [USDA warn] {e}")
        return None

def _get_fallback(name: str) -> NutritionData:
    """Guess a generic category from the ingredient name."""
    n = name.lower()
    if any(w in n for w in ["oil", "butter", "lard"]): return FALLBACK_GENERICS["oil"]
    if any(w in n for w in ["chicken", "beef", "pork", "fish", "salmon", "tuna", "turkey", "lamb"]): return FALLBACK_GENERICS["meat"]
    if any(w in n for w in ["milk", "cheese", "yogurt", "cream", "dairy"]): return FALLBACK_GENERICS["dairy"]
    if any(w in n for w in ["rice", "pasta", "bread", "oat", "grain", "flour", "cereal"]): return FALLBACK_GENERICS["grain"]
    if any(w in n for w in ["apple", "banana", "orange", "berry", "grape", "mango", "fruit"]): return FALLBACK_GENERICS["fruit"]
    if any(w in n for w in ["broccoli", "spinach", "carrot", "lettuce", "tomato", "pepper", "onion", "vegetable"]): return FALLBACK_GENERICS["vegetable"]
    return FALLBACK_GENERICS["default"]

print("✓ USDA nutrition lookup helpers defined")

✓ USDA nutrition lookup helpers defined


In [13]:
# --- Cross-check: USDA API results vs real values ---
# Uses _usda_search (with smart scoring) + sleeps to avoid DEMO_KEY rate limit
import time

CHECKS = [
    # (query,           amount, unit,    real_kcal, label)
    ("banana",           1,      "piece",  105,  "1 medium banana"),
    ("greek yogurt",     1,      "cup",    130,  "1 cup plain greek yogurt"),
    ("honey",            1,      "tbsp",    64,  "1 tbsp honey"),
    ("chicken breast",  100,     "g",      165,  "100g grilled chicken breast"),
    ("white rice",      100,     "g",      130,  "100g cooked white rice"),
    ("olive oil",        1,      "tbsp",   119,  "1 tbsp olive oil"),
]

print(f"{'Food':<26} {'USDA match picked':<40} {'per100g':<10} {'scaled':<10} {'real':<8} diff")
print("-" * 105)

for query, amount, unit, real_kcal, label in CHECKS:
    # Fetch raw results with same params as _usda_search to show which food was picked
    try:
        r = requests.get(
            USDA_SEARCH_URL,
            params={"query": query, "pageSize": 20, "api_key": USDA_API_KEY,
                    "dataType": ["Foundation", "SR Legacy"]},
            timeout=10,
        )
        foods = r.json().get("foods", [])
    except Exception as e:
        print(f"{label:<26} ERROR: {e}")
        time.sleep(3)
        continue

    if not foods:
        print(f"{label:<26} NO USDA RESULT (rate limited or no match)")
        time.sleep(3)
        continue

    # Replicate _usda_search scoring to show which item was actually picked
    q_words = query.lower().split()
    def _score(food):
        desc = food.get("description", "").lower()
        hits    = sum(1 for w in q_words if w in desc)
        first   = 2 if any(desc.startswith(w) for w in q_words) else 0
        penalty = -3 * len(_PROCESSED & set(desc.split(",")[0].split()))
        return (first + hits + penalty, -len(desc))

    best      = max(foods, key=_score)
    top_name  = best.get("description", "?")[:38]
    base      = _extract_nutrients(best)
    grams     = unit_to_grams(amount, unit)
    scaled    = _scale(base, grams)
    diff      = scaled.calories - real_kcal
    flag      = "✓" if abs(diff) < 25 else f"✗ ({diff:+.0f})"

    print(f"{label:<26} {top_name:<40} {base.calories:<10.1f} {scaled.calories:<10.1f} {real_kcal:<8} {flag}")
    time.sleep(3)   # DEMO_KEY: max ~3 req/min

print("\nDone.")


Food                       USDA match picked                        per100g    scaled     real     diff
---------------------------------------------------------------------------------------------------------
1 medium banana            Bananas, raw                             89.0       89.0       105      ✓
1 cup plain greek yogurt   Yogurt, Greek, plain, lowfat             73.0       175.2      130      ✗ (+45)
1 tbsp honey               Honey                                    304.0      45.6       64       ✓
100g grilled chicken breast Chicken breast, roll, oven-roasted       134.0      134.0      165      ✗ (-31)
100g cooked white rice     Rice, white, steamed, Chinese restaura   151.0      151.0      130      ✓
1 tbsp olive oil           Oil, olive, extra light                  0.0        0.0        119      ✗ (-119)

Done.


In [14]:
def _cache_get(conn: sqlite3.Connection, name: str) -> NutritionData | None:
    row = conn.execute(
        "SELECT nutrition_data FROM ingredient_cache WHERE ingredient_name=?", (name.lower(),)
    ).fetchone()
    if row:
        d = json.loads(row[0])
        return NutritionData(**d)
    return None

def _cache_set(conn: sqlite3.Connection, name: str, nutrition: NutritionData) -> None:
    conn.execute(
        """INSERT OR REPLACE INTO ingredient_cache (ingredient_name, amount_value, unit, nutrition_data, cached_at)
           VALUES (?, 100, 'g', ?, CURRENT_TIMESTAMP)""",
        (name.lower(), json.dumps(nutrition.__dict__)),
    )
    conn.commit()

def nutrition_lookup_agent(ingredients: list[Ingredient], conn: sqlite3.Connection) -> list[Ingredient]:
    """
    Enrich each Ingredient with NutritionData. Uses cache first, then USDA, then generic fallback.
    Mutates each Ingredient in-place and returns the list.
    """
    for ing in ingredients:
        grams = unit_to_grams(ing.amount, ing.unit)
        if grams <= 0:
            grams = 100.0  # unknown amount → assume 100g default serving
        query = ing.name + (f" {ing.cooking_method}" if ing.cooking_method else "")
        
        # 1. Check cache
        cached = _cache_get(conn, ing.name)
        if cached:
            ing.nutrition = _scale(cached, grams)
            print(f"  [cache] {ing.name}: {ing.nutrition.calories} kcal")
            continue
        
        # 2. USDA API
        base = _usda_search(query)
        if base:
            _cache_set(conn, ing.name, base)
            ing.nutrition = _scale(base, grams)
            print(f"  [USDA]  {ing.name}: {ing.nutrition.calories} kcal")
        else:
            # 3. Generic fallback
            base = _get_fallback(ing.name)
            ing.nutrition = _scale(base, grams)
            ing.confidence = min(ing.confidence, 0.4)
            ing.source = "fallback_average"
            print(f"  [fallback] {ing.name}: {ing.nutrition.calories} kcal (generic)")
    
    return ingredients

print("✓ Nutrition lookup agent defined")

✓ Nutrition lookup agent defined


# 7 — Database Agent (SQLite)

Creates the schema and provides functions to persist meals and ingredients. Also sets up the ingredient cache table used by the nutrition lookup agent.

In [15]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS meals (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id             TEXT DEFAULT 'local',
    meal_timestamp      DATETIME,
    meal_date           DATE,
    meal_type           TEXT,
    mood                TEXT,
    notes               TEXT,
    total_calories      REAL,
    total_protein_g     REAL,
    total_fat_g         REAL,
    total_carbs_g       REAL,
    total_fiber_g       REAL,
    api_fallback_count  INTEGER DEFAULT 0,
    created_at          DATETIME DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS meal_ingredients (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    meal_id         INTEGER,
    ingredient_name TEXT,
    amount_value    REAL,
    unit            TEXT,
    cooking_method  TEXT,
    calories        REAL,
    protein_g       REAL,
    fat_g           REAL,
    carbs_g         REAL,
    fiber_g         REAL,
    api_source      TEXT,
    confidence      REAL,
    source          TEXT,
    FOREIGN KEY (meal_id) REFERENCES meals(id)
);

CREATE TABLE IF NOT EXISTS ingredient_cache (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    ingredient_name TEXT UNIQUE,
    amount_value    INTEGER,
    unit            TEXT,
    nutrition_data  JSON,
    cached_at       DATETIME
);
"""

def get_db() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.executescript(SCHEMA)
    return conn

def save_meal(meal: Meal, conn: sqlite3.Connection) -> int:
    """Insert a Meal + its ingredients and return the meal id."""
    cur = conn.execute(
        """INSERT INTO meals
           (meal_timestamp, meal_date, meal_type, mood, notes,
            total_calories, total_protein_g, total_fat_g, total_carbs_g, total_fiber_g, api_fallback_count)
           VALUES (?,?,?,?,?,?,?,?,?,?,?)""",
        (
            meal.meal_timestamp.isoformat(), meal.meal_date.isoformat(), meal.meal_type,
            meal.mood, meal.notes,
            meal.total_calories, meal.total_protein_g, meal.total_fat_g,
            meal.total_carbs_g, meal.total_fiber_g, meal.api_fallback_count,
        ),
    )
    meal_id = cur.lastrowid
    
    for ing in meal.ingredients:
        n = ing.nutrition
        conn.execute(
            """INSERT INTO meal_ingredients
               (meal_id, ingredient_name, amount_value, unit, cooking_method,
                calories, protein_g, fat_g, carbs_g, fiber_g, api_source, confidence, source)
               VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)""",
            (
                meal_id, ing.name, ing.amount, ing.unit, ing.cooking_method,
                n.calories if n else 0, n.protein_g if n else 0, n.fat_g if n else 0,
                n.carbs_g if n else 0, n.fiber_g if n else 0,
                n.api_source if n else "none", ing.confidence, ing.source,
            ),
        )
    conn.commit()
    meal.id = meal_id
    return meal_id

def get_daily_summary(target_date: date, conn: sqlite3.Connection) -> dict:
    row = conn.execute(
        """SELECT SUM(total_calories), SUM(total_protein_g), SUM(total_fat_g),
                  SUM(total_carbs_g), COUNT(*)
           FROM meals WHERE meal_date=?""",
        (target_date.isoformat(),),
    ).fetchone()
    return {
        "date": target_date.isoformat(),
        "total_calories": row[0] or 0,
        "total_protein_g": row[1] or 0,
        "total_fat_g": row[2] or 0,
        "total_carbs_g": row[3] or 0,
        "meal_count": row[4] or 0,
    }

print("✓ Database agent defined")

✓ Database agent defined


In [16]:
# --- Test: Database Agent ---
import os

# Use a temp db for this test
test_db = Path("cotd_test.db")
if test_db.exists():
    test_db.unlink()

conn = sqlite3.connect(test_db)
conn.executescript(SCHEMA)

# Build a small meal manually
n1 = NutritionData(calories=245, protein_g=45.0, fat_g=5.0, carbs_g=0.0, api_source="USDA_FDC")
n2 = NutritionData(calories=26, protein_g=2.4, fat_g=0.4, carbs_g=4.3, fiber_g=0.9, api_source="USDA_FDC")
ing1 = Ingredient("chicken breast", 145, "g", cooking_method="grilled", nutrition=n1)
ing2 = Ingredient("broccoli", 90, "g", cooking_method="steamed", nutrition=n2)

meal = Meal(
    meal_type="lunch",
    ingredients=[ing1, ing2],
    total_calories=271,
    total_protein_g=47.4,
    total_fat_g=5.4,
    total_carbs_g=4.3,
)
meal_id = save_meal(meal, conn)

# Verify
row = conn.execute("SELECT total_calories, meal_type FROM meals WHERE id=?", (meal_id,)).fetchone()
ings = conn.execute("SELECT ingredient_name FROM meal_ingredients WHERE meal_id=?", (meal_id,)).fetchall()
assert row[0] == 271 and row[1] == "lunch", f"Unexpected row: {row}"
assert len(ings) == 2, f"Expected 2 ingredients, got {len(ings)}"

conn.close()
test_db.unlink()   # cleanup
print(f"✓ Database test passed — saved meal id={meal_id} with {len(ings)} ingredients")

✓ Database test passed — saved meal id=1 with 2 ingredients


# 8 — End-to-End Orchestrator

Ties all agents together. Handles the interactive questioning loop: asks the reasoning agent if we need more info, prompts the user (in notebook via `input()`), then proceeds once completeness ≥ 0.75 or the user skips.

In [17]:
MAX_QUESTIONS = 3   # cap the agentic questioning loop

def _infer_meal_type() -> str:
    hour = datetime.now().hour
    if hour < 10:  return "breakfast"
    if hour < 14:  return "lunch"
    if hour < 18:  return "snack"
    return "dinner"

def _apply_user_clarification(ingredients: list[Ingredient], user_reply: str, question: str) -> list[Ingredient]:
    """
    Parse clarification reply and update matching ingredients.
    Falls back to regex extraction if the LLM parse fails.
    """
    import re as _re
    prompt = (
        f"A user was asked: '{question}'\n"
        f"They replied: '{user_reply}'\n\n"
        "Extract any food amounts or cooking methods mentioned. "
        "Respond ONLY with JSON like:\n"
        '{"updates": [{"name": "...", "amount": ..., "unit": "...", "cooking_method": "..."}]}'
    )
    applied = False
    try:
        reply = ollama_chat(TEXT_AGENT_MODEL, [{"role": "user", "content": prompt}])
        updates = extract_json(reply).get("updates", [])
        for upd in updates:
            for ing in ingredients:
                if upd["name"].lower() in ing.name.lower() or ing.name.lower() in upd["name"].lower():
                    if upd.get("amount"): ing.amount = float(upd["amount"]); applied = True
                    if upd.get("unit"):   ing.unit = upd["unit"]
                    if upd.get("cooking_method"): ing.cooking_method = upd["cooking_method"]
                    ing.source = "user_input"; ing.confidence = 0.9
    except Exception:
        pass

    # Regex fallback: if LLM extracted nothing, pull a number + unit from the reply
    if not applied:
        m = _re.search(r'(\d+(?:\.\d+)?)\s*(g|kg|ml|l|cup|tbsp|tsp|piece|oz|lb)?', user_reply, _re.IGNORECASE)
        if m:
            amount = float(m.group(1))
            unit   = (m.group(2) or "g").lower()
            # Apply to the ingredient with the lowest amount (most likely what was asked)
            target = min(ingredients, key=lambda i: i.amount)
            target.amount = amount; target.unit = unit
            target.source = "user_input"; target.confidence = 0.85
    return ingredients

def log_meal(
    user_message: str,
    meal_type: str | None = None,
    interactive: bool = True,
) -> Meal:
    """
    Full pipeline:
      1. Route input
      2. Parse ingredients
      3. Reasoning loop (ask questions if needed)
      4. Nutrition lookup
      5. Save to DB
      6. Return Meal with summary
    """
    conn = get_db()
    
    # ── 1. Route ──────────────────────────────────────────────────────────────
    route = router_agent(user_message)
    print(f"\n[Router] → {route}")
    
    # ── 2. Parse ──────────────────────────────────────────────────────────────
    if route == "TEXT_AGENT":
        ingredients = text_agent(user_message)
    else:
        # IMAGE_AGENT: placeholder (no vision model available in current setup)
        print("[Image Agent] Vision model not loaded — please describe ingredients in text.")
        description = input("Describe what you see in the photo: ").strip() if interactive else user_message
        ingredients = text_agent(description)
    
    print(f"[Text Agent] Parsed {len(ingredients)} ingredient(s):")
    for ing in ingredients:
        print(f"  • {ing.name}: {ing.amount} {ing.unit}  method={ing.cooking_method}")
    
    # ── 3. Reasoning / questioning loop ───────────────────────────────────────
    asked_questions = set()   # prevent asking the same question twice
    for turn in range(MAX_QUESTIONS):
        check = reasoning_agent(ingredients)
        score = check.get("completeness_score", 0.0)
        question = check.get("next_question")

        print(f"\n[Reasoning] completeness={score:.2f}  question={question}")

        if score >= 0.75 or not question:
            break  # complete enough — proceed

        if not interactive:
            print("  [non-interactive] skipping clarification, using defaults")
            break

        # Don't repeat the same question
        q_key = question.lower()[:60]
        if q_key in asked_questions:
            print("  [loop guard] already asked this — moving forward")
            break
        asked_questions.add(q_key)

        # Ask → Wait → decide
        print(f"\n🤖 {question}")
        user_reply = input("Your answer (or press Enter to skip): ").strip()

        if not user_reply:
            # User skipped → mark unknowns as fallback, stop asking
            print("  [skipped] using default values for uncertain ingredients")
            for ing in ingredients:
                if ing.amount <= 0:
                    ing.source = "fallback_average"
            break

        # User answered → apply and remember it was handled; loop re-checks score
        before = {i.name: i.amount for i in ingredients}
        ingredients = _apply_user_clarification(ingredients, user_reply, question)
        after  = {i.name: i.amount for i in ingredients}
        changed = [n for n in before if before[n] != after[n]]
        if changed:
            print(f"  [updated] {', '.join(changed)}")
        else:
            print("  [warning] could not parse answer — using defaults for uncertain items")
            for ing in ingredients:
                if ing.amount <= 0:
                    ing.amount = 100.0; ing.unit = "g"; ing.source = "fallback_average"
            break
    
    # ── 4. Nutrition lookup ────────────────────────────────────────────────────
    print("\n[Nutrition Lookup (per 100g)]")
    ingredients = nutrition_lookup_agent(ingredients, conn)
    
    # ── 5. Aggregate & save ────────────────────────────────────────────────────
    meal = Meal(
        meal_type=meal_type or _infer_meal_type(),
        ingredients=ingredients,
        total_calories=sum(i.nutrition.calories for i in ingredients if i.nutrition),
        total_protein_g=sum(i.nutrition.protein_g for i in ingredients if i.nutrition),
        total_fat_g=sum(i.nutrition.fat_g for i in ingredients if i.nutrition),
        total_carbs_g=sum(i.nutrition.carbs_g for i in ingredients if i.nutrition),
        total_fiber_g=sum((i.nutrition.fiber_g or 0) for i in ingredients if i.nutrition),
        api_fallback_count=sum(1 for i in ingredients if i.source == "fallback_average"),
    )
    
    meal_id = save_meal(meal, conn)
    conn.close()
    
    # ── 6. Summary ────────────────────────────────────────────────────────────
    print(f"""
╔══════════════════════════════════════╗
  Meal logged! (id={meal_id}, {meal.meal_type})
  🔥 Calories:  {meal.total_calories:.0f} kcal
  💪 Protein:   {meal.total_protein_g:.1f} g
  🥑 Fat:       {meal.total_fat_g:.1f} g
  🌾 Carbs:     {meal.total_carbs_g:.1f} g
  🥦 Fiber:     {meal.total_fiber_g:.1f} g
  Fallbacks used: {meal.api_fallback_count}
╚══════════════════════════════════════╝""")
    
    return meal

print("✓ Orchestrator defined")

✓ Orchestrator defined


In [18]:
# --- Test: Interactive questioning loop (mocked input) ---
# unittest.mock.patch replaces builtins.input with a scripted function,
# letting us test the ask→wait→answer flow without typing anything.
from unittest.mock import patch

def make_input(*answers):
    """Returns a fake input() that plays back `answers` then returns '' (skip)."""
    it = iter(answers)
    def _fake(prompt=""):
        try:
            reply = next(it)
            print(f"[mock input] {reply!r}")
            return reply
        except StopIteration:
            print("[mock input] '' (auto-skip remaining)")
            return ""
    return _fake

# ── Test 1: user answers the question ──────────────────────────────────────
print("=" * 50)
print("TEST 1: user answers with quantity")
print("=" * 50)

with patch("builtins.input", make_input("200g of pasta, 100g tomato sauce")):
    meal = log_meal(
        "I had some pasta and tomato sauce",
        meal_type="dinner",
        interactive=True,
    )

assert meal.total_calories > 0, "Should have calories after user answered"
print(f"\n✓ Test 1 passed — {meal.total_calories:.0f} kcal logged")

# ── Test 2: user skips (presses Enter) ─────────────────────────────────────
print()
print("=" * 50)
print("TEST 2: user skips every question")
print("=" * 50)

with patch("builtins.input", make_input()):   # no scripted answers → always ""
    meal2 = log_meal(
        "I had some pasta and tomato sauce",
        meal_type="dinner",
        interactive=True,
    )

assert meal2.total_calories > 0, "Should still have calories from 100g defaults"
print(f"\n✓ Test 2 passed — {meal2.total_calories:.0f} kcal (default 100g portions)")


TEST 1: user answers with quantity

[Router] → TEXT_AGENT
[Text Agent] Parsed 2 ingredient(s):
  • pasta: 0.0 unknown  method=None
  • tomato sauce: 0.0 unknown  method=None

[Reasoning] completeness=0.30  question=How did you measure your pasta? (e.g. cups, grams, or pieces)

🤖 How did you measure your pasta? (e.g. cups, grams, or pieces)
[mock input] '200g of pasta, 100g tomato sauce'

[Reasoning] completeness=0.30  question=How did you measure your pasta? (e.g. cups, grams, or pieces)

🤖 How did you measure your pasta? (e.g. cups, grams, or pieces)
[mock input] '' (auto-skip remaining)
  [skipped] using default values for uncertain ingredients

[Nutrition Lookup (per 100g)]
  [cache] pasta: 371.0 kcal
  [cache] tomato sauce: 95.0 kcal

╔══════════════════════════════════════╗
  Meal logged! (id=5, dinner)
  🔥 Calories:  466 kcal
  💪 Protein:   14.2 g
  🥑 Fat:       1.7 g
  🌾 Carbs:     96.7 g
  🥦 Fiber:     4.7 g
  Fallbacks used: 2
╚══════════════════════════════════════╝

✓ Test 1

# 9 — End-to-End Tests (Non-Interactive)

Two integration tests that run the full pipeline without waiting for user input, verifying the happy path and the fallback path.

In [19]:
# --- Test: End-to-End (Text, complete, non-interactive) ---
print("=" * 50)
print("TEST 1: Smoothie (complete text description)")
print("=" * 50)

meal1 = log_meal(
    "Had a morning smoothie: 1 banana, 1 cup Greek yogurt, 1 tbsp honey",
    meal_type="breakfast",
    interactive=False,
)

assert meal1.total_calories > 0, "Calories should be > 0"
assert meal1.id is not None, "Meal should have a DB id"
assert len(meal1.ingredients) >= 2, "Should parse ≥ 2 ingredients"
print(f"\n✓ Test 1 passed — id={meal1.id}, {meal1.total_calories:.0f} kcal")

TEST 1: Smoothie (complete text description)

[Router] → TEXT_AGENT
[Text Agent] Parsed 3 ingredient(s):
  • banana: 1.0 piece  method=None
  • greek yogurt: 1.0 cup  method=None
  • honey: 1.0 tbsp  method=None

[Reasoning] completeness=0.90  question=None

[Nutrition Lookup (per 100g)]
  [cache] banana: 89.0 kcal
  [cache] greek yogurt: 175.2 kcal
  [cache] honey: 45.6 kcal

╔══════════════════════════════════════╗
  Meal logged! (id=7, breakfast)
  🔥 Calories:  310 kcal
  💪 Protein:   25.0 g
  🥑 Fat:       4.9 g
  🌾 Carbs:     44.6 g
  🥦 Fiber:     2.6 g
  Fallbacks used: 0
╚══════════════════════════════════════╝

✓ Test 1 passed — id=7, 310 kcal


In [21]:
# --- Daily Summary ---
conn = get_db()
summary = get_daily_summary(date.today(), conn)
conn.close()

print("Today's totals:")
print(f"  Meals logged: {summary['meal_count']}")
print(f"  Calories:     {summary['total_calories']:.0f} kcal")
print(f"  Protein:      {summary['total_protein_g']:.1f} g")
print(f"  Fat:          {summary['total_fat_g']:.1f} g")
print(f"  Carbs:        {summary['total_carbs_g']:.1f} g")

Today's totals:
  Meals logged: 8
  Calories:     3416 kcal
  Protein:      135.5 g
  Fat:          20.3 g
  Carbs:        669.4 g


# 10 — Interactive Usage

Run this cell to log a meal interactively. The agent will ask clarifying questions if your description is incomplete.

```python
meal = log_meal("your food description here", interactive=True)
```